# Chapter 3 — The Reason–Act–Observe Loop

This lab runs one task through ReAct-shaped and CodeAct-shaped loops, then
demonstrates why correct observations do not guarantee a correct answer.

The model output is scripted for determinism. Tool calls and code execution are
real within the in-memory teaching environment in
[`../code/react_vs_codeact.py`](../code/react_vs_codeact.py).

In [1]:
import sys
sys.path.insert(0, "../code")

from react_vs_codeact import (
    TASK_PROMPT, REACT_SCRIPT, ScriptedReActModel, run_react_loop,
    CODEACT_THOUGHT, CODEACT_CODE, run_codeact_loop, read_file,
    render_timeline, LONG_WORKSPACE, CORRECT_LONG_REACT_SCRIPT,
    FLAWED_LONG_REACT_SCRIPT, run_long_react_loop, run_long_codeact_loop,
)

print(TASK_PROMPT)

Which of a.txt, b.txt, c.txt contains the largest number, and what is that number plus 10?


## 1. Same loop, different action shape

`run_react_loop` calls `read_file` once per action. `run_codeact_loop`
executes one fixed code block. Both produce the same answer, but composition
lives in different parts of the system.

In [2]:
react_transcript = run_react_loop(ScriptedReActModel(REACT_SCRIPT))
for kind, text in react_transcript:
    print(f"{kind}: {text}")

n_actions = sum(1 for kind, _ in react_transcript if kind == "Action")
print(f"\nReAct-shaped trace: {n_actions} actions, {len(react_transcript)} entries")

Thought: I need each file's value before I can compare them. Start with a.txt.
Action: read_file('a.txt')
Observation: 42
Thought: a.txt is 42. Now check b.txt.
Action: read_file('b.txt')
Observation: 17
Thought: b.txt is 17, smaller than 42 so far. Now check c.txt.
Action: read_file('c.txt')
Observation: 8
Thought: c.txt is 8. The largest value seen is 42, from a.txt. 42 + 10 = 52.
Finish: a.txt has the largest number (42); 42 + 10 = 52.

ReAct-shaped trace: 3 actions, 11 entries


In [3]:
codeact_transcript = run_codeact_loop(
    CODEACT_THOUGHT, CODEACT_CODE, {"read_file": read_file}
)
for kind, text in codeact_transcript:
    print(f"{kind}: {text}")

n_actions = sum(1 for kind, _ in codeact_transcript if kind.startswith("Action"))
print(f"\nCodeAct-shaped trace: {n_actions} action, {len(codeact_transcript)} entries")

Thought: I'll read all three files, compare them, and compute the answer in one action.
Action (code): values = {f: int(read_file(f)) for f in ["a.txt", "b.txt", "c.txt"]}
largest_file = max(values, key=values.get)
result = values[largest_file] + 10
print(f"{largest_file} has the largest number ({values[largest_file]}); "
      f"{values[largest_file]} + 10 = {result}")
Observation: a.txt has the largest number (42); 42 + 10 = 52
Finish: a.txt has the largest number (42); 42 + 10 = 52

CodeAct-shaped trace: 1 action, 4 entries


## 2. Correct observations can still lead to a wrong answer

The next two six-file scripts receive identical, correct observations. They
differ at one reasoning step, where the flawed script treats `55 > 42` as
false. This isolates the difference between correct environment feedback and
correct interpretation of that feedback.

In [4]:
expected_file = max(LONG_WORKSPACE, key=lambda path: int(LONG_WORKSPACE[path]))
expected_value = int(LONG_WORKSPACE[expected_file])

print(f"Ground truth: {expected_file} = {expected_value}; result = {expected_value + 10}")
print("\nCorrect decision:")
print(CORRECT_LONG_REACT_SCRIPT[4].thought)
print("\nFlawed decision:")
print(FLAWED_LONG_REACT_SCRIPT[4].thought)

Ground truth: d.txt = 55; result = 65

Correct decision:
d.txt is 55. 55 > 42, so the new max is 55 (d.txt). Check e.txt.

Flawed decision:
d.txt is 55. That's less than 42, so max stays 42 (a.txt). Check e.txt.


In [5]:
correct_trace = run_long_react_loop(CORRECT_LONG_REACT_SCRIPT)
flawed_trace = run_long_react_loop(FLAWED_LONG_REACT_SCRIPT)

print(f"Correct trace: {correct_trace[-1][1]}")
print(f"Flawed trace:  {flawed_trace[-1][1]}")
print("\nBoth traces received correct values from every read_file action.")

Correct trace: d.txt has the largest number (55); 55 + 10 = 65.
Flawed trace:  a.txt has the largest number (42); 42 + 10 = 52.

Both traces received correct values from every read_file action.


The environment is correct, but the flawed script's state is not. The loop
checks that each action executes; it has no postcondition that validates the
selected maximum. Successful actions therefore do not prove task success.

## 3. Delegate deterministic computation

The CodeAct-shaped version delegates comparison to Python's `max()` instead of
tracking a running value in prose.

In [6]:
code_trace, code_answer = run_long_codeact_loop()
for kind, text in code_trace:
    print(f"{kind}: {text}")
print(f"\nAnswer: {code_answer}")

Thought: I'll read all six files and let Python's max() find the largest — no manual tracking.
Action (code): values = {f: int(read_file(f)) for f in ["a.txt", "b.txt", "c.txt", "d.txt", "e.txt", "f.txt"]}
largest_file = max(values, key=values.get)
result = values[largest_file] + 10
print(f"{largest_file} has the largest number ({values[largest_file]}); {values[largest_file]} + 10 = {result}.")
Observation: d.txt has the largest number (55); 55 + 10 = 65.
Finish: d.txt has the largest number (55); 55 + 10 = 65.

Answer: d.txt has the largest number (55); 55 + 10 = 65.


This makes the comparison explicit and testable; it does not make generated
code automatically correct. Replacing `max()` with `min()`, omitting a file,
or parsing values incorrectly would still produce a wrong result. Reliable
loops verify outcomes, not merely successful execution.

## 4. Conceptual lineage

ReAct, PAL, Toolformer, and CodeAct address related but distinct questions:
interleaved acting, interpreter-aided computation, learned API use, and
executable code actions.

In [7]:
print(render_timeline())

2022-10-06  ReAct
           Yao, Zhao, Yu, Du, Shafran, Narasimhan, Cao — arXiv:2210.03629
           Interleaves reasoning traces with task-specific environment actions.

2022-11-18  PAL
           Gao, Madaan, Zhou, Alon, Liu, Yang, Callan, Neubig — arXiv:2211.10435
           Generates programs as intermediate reasoning and delegates computation to an interpreter.

2023-02-09  Toolformer
           Schick, Dwivedi-Yu, Dessi, Raileanu, Lomeli, Zettlemoyer, Cancedda, Scialom — arXiv:2302.04761
           Trains a model to decide when and how to call external APIs and use their results.

2024-02-01  CodeAct
           Wang, Chen, Yuan, Zhang, Li, Peng, Ji — arXiv:2402.01030
           Uses executable Python as a unified action space within an iterative agent that can revise actions after new observations.

